# 🛒 Modul 07: Unsupervised Learning II - Aturan Asosiasi (*Association Rule Mining*)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/07_Association_Rule_Mining.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & analogi dunia nyata sebelum rumus matematika dan kode.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── [5. Evaluation] ◄── 🎯 [4. MODELING: ASSOCIATION RULES]
```
Di **Modul 06**, kita mengelompokkan pelanggan ke dalam segmen. Sekarang di **Modul 07**, kita membedah isi keranjang belanja mereka: **Barang apa saja yang cenderung dibeli bersamaan (*Market Basket Analysis*)** untuk menyusun strategi promo bundling dan penataan rak supermarket!


## 🧺 1. Filosofi & Kisah Nyata: Market Basket Analysis

### 👶 Kisah Klasik Supermarket: Popok Bayi & Minuman Kaleng
Salah satu legenda paling terkenal dalam sejarah *Data Mining* berasal dari sebuah jaringan supermarket di Amerika Serikat:
* Ketika data jutaan struk kasir dianalisis, para analis menemukan pola belanja aneh yang tidak pernah terpikirkan sebelumnya: **Pada hari Jumat sore antara pukul 17.00 - 19.00, pembelian Popok Bayi (*Diapers*) sangat sering dibarengi dengan pembelian Minuman Kaleng (*Beer*)!**
* **Mengapa demikian?** Ternyata, para ayah muda diminta oleh istrinya untuk membeli popok bayi sepulang kerja menjelang akhir pekan. Saat berada di lorong kasir, para ayah tersebut memutuskan untuk sekaligus mengambil minuman kaleng untuk bersantai menonton pertandingan sepak bola di akhir pekan.
* **Tindakan Bisnis**: Supermarket meletakkan rak minuman kaleng tepat di sebelah rak popok bayi pada hari Jumat, dan penjualan keduanya langsung melonjak drastis!

### 💼 Aplikasi Aturan Asosiasi di Dunia Modern:
1. **Penataan Rak Fisik (*Cross-Merchandising*)**: Menaruh kecap dan saus tiram di dekat rak daging potong.
2. **Paket Diskon (*Bundling Promo*)**: Paket Kombo Bioskop (Popcorn + Minuman Soda).
3. **Sistem Rekomendasi E-Commerce**: Fitur *"Pelanggan yang membeli produk ini juga membeli..."* di Amazon, Shopee, dan Tokopedia.


## 📐 2. Membedah 3 Pilar Metrik Asosiasi: Support, Confidence, dan Lift

Aturan asosiasi ditulis dalam format:  
$$\text{Antecedent (A)} \longrightarrow \text{Consequent (B)}$$
*(Artinya: "Jika seseorang membeli A, maka ia cenderung membeli B")*

Untuk memisahkan aturan emas yang menguntungkan dari aturan sampah yang menyesatkan, kita menggunakan 3 metrik:

---

### 1. Support (Tingkat Kepopuleran)
$$\text{Support}(A \rightarrow B) = \frac{\text{Jumlah Struk yang Membeli A dan B Bersamaan}}{\text{Total Seluruh Struk Kasir}}$$
* **Analogi**: *"Seberapa sering kombinasi barang ini muncul di seluruh transaksi toko?"*  
  Jika dari 1.000 transaksi ada 200 transaksi yang membeli Roti dan Mentega bersamaan, maka Support-nya adalah $20\%$.

---

### 2. Confidence (Tingkat Kepastian / Keyakinan)
$$\text{Confidence}(A \rightarrow B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$$
* **Analogi**: *"Dari 100 orang yang sudah meletakkan Roti di keranjang belanjanya, berapa orang yang PASTI ikut mengambil Mentega?"*  
  Jika 80 orang di antaranya ikut mengambil Mentega, maka Confidence-nya adalah $80\%$.

---

### 3. Lift Ratio (Kekuatan Hubungan Nyata vs Kebetulan)
$$\text{Lift}(A \rightarrow B) = \frac{\text{Confidence}(A \rightarrow B)}{\text{Support}(B)}$$
* **Analogi**: *"Apakah pembelian B benar-benar dipicu oleh A, ataukah barang B memang laku keras secara alami?"*
  * **Lift = 1**: **Tidak Ada Hubungan**. Barang A dan B dibeli bersamaan murni karena faktor kebetulan acak (independen).
  * **Lift > 1**: **Hubungan Positif Nyata**! Membeli barang A secara signifikan menaikkan kemungkinan membeli barang B. Semakin tinggi nilai Lift di atas 1, semakin kuat ikatannya!
  * **Lift < 1**: **Hubungan Negatif**. Membeli barang A justru menurunkan minat membeli B (misal: dua merek kopi instan yang saling kanibal).


## 🧾 3. Menyiapkan Log Transaksi & Transformasi One-Hot Encoding

Di dunia nyata, data kasir berbentuk daftar keranjang transaksi (*List of Lists*).  
Pustaka data mining memerlukan data ini dalam bentuk **Matriks Biner One-Hot (True/False)**, di mana:
* Baris = 1 Struk Belanja Pelanggan.
* Kolom = Nama Produk.
* Nilai = `True` jika produk dibeli, `False` jika tidak.


In [ ]:
# Prapemrosesan dan Pengecekan Pustaka mlxtend
try:
    import mlxtend
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlxtend", "-q"])
    import mlxtend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# Simulasi 12 Log Struk Transaksi Kasir Supermarket Lokal
log_transaksi = [
    ['Roti Tawar', 'Mentega', 'Susu Sapi'],
    ['Roti Tawar', 'Selai Cokelat'],
    ['Roti Tawar', 'Mentega', 'Keju Cheddar'],
    ['Kopi Bubuk', 'Gula Pasir'],
    ['Roti Tawar', 'Mentega', 'Susu Sapi', 'Keju Cheddar'],
    ['Kopi Bubuk', 'Gula Pasir', 'Susu Sapi'],
    ['Roti Tawar', 'Mentega'],
    ['Roti Tawar', 'Selai Cokelat', 'Susu Sapi'],
    ['Kopi Bubuk', 'Gula Pasir', 'Biskuit'],
    ['Roti Tawar', 'Mentega', 'Selai Cokelat'],
    ['Susu Sapi', 'Biskuit'],
    ['Kopi Bubuk', 'Gula Pasir', 'Susu Sapi', 'Biskuit']
]

print(f"Total Struk Kasir: {len(log_transaksi)} transaksi")
print("Contoh 3 Struk Pertama:")
for i, t in enumerate(log_transaksi[:3], 1):
    print(f" Struk {i}: {', '.join(t)}")

# Transformasi Log Transaksi menjadi Matriks Biner One-Hot
te = TransactionEncoder()
te_ary = te.fit(log_transaksi).transform(log_transaksi)
df_transaksi = pd.DataFrame(te_ary, columns=te.columns_)

print("\nBentuk Matriks Biner Transaksi Kasir:")
df_transaksi.head()


## 🌳 4. Menemukan Kombinasi Laris: Apriori vs FP-Growth

Sebelum membuat aturan, kita harus mencari himpunan barang yang sering dibeli bersamaan (*Frequent Itemsets*) dengan batas **Minimum Support**.

---

### A. Algoritma Apriori (*The Golden Rule of Pruning*)
* **Prinsip Apriori**: *"Jika suatu kombinasi barang tidak populer (Support rendah), maka SEMUA kombinasi turunannya yang lebih besar PASTI tidak populer!"*  
* *Contoh*: Jika tak ada seorang pun yang membeli Payung di musim kemarau, maka tidak perlu membuang waktu menghitung apakah ada orang yang membeli `[Payung, Kopi, Roti]`. Cabang itu langsung dipangkas (*Pruned*)!

---

### B. Algoritma FP-Growth (*Frequent Pattern Tree*)
* Apriori memindai seluruh basis data berulang kali untuk setiap penambahan kombinasi barang.
* **FP-Growth** jauh lebih cepat karena mengompresi log transaksi menjadi satu struktur pohon padat (*FP-Tree*). FP-Growth dapat mengekstrak pola 10x hingga 100x lebih cepat pada basis data raksasa jutaan transaksi!


In [ ]:
# 1. Ekstraksi Frequent Itemsets menggunakan Apriori (Min Support = 25% atau muncul di minimal 3 struk)
itemset_apriori = apriori(df_transaksi, min_support=0.25, use_colnames=True)
itemset_apriori['jumlah_item'] = itemset_apriori['itemsets'].apply(lambda x: len(x))
itemset_apriori = itemset_apriori.sort_values(by='support', ascending=False)

# 2. Ekstraksi Menggunakan FP-Growth
itemset_fpgrowth = fpgrowth(df_transaksi, min_support=0.25, use_colnames=True)

print(f"Jumlah Frequent Itemsets Ditemukan (Apriori)   : {len(itemset_apriori)}")
print(f"Jumlah Frequent Itemsets Ditemukan (FP-Growth) : {len(itemset_fpgrowth)}")
print("\nTop 6 Kombinasi Barang Paling Sering Dibeli:")
display_df = itemset_apriori[['itemsets', 'support', 'jumlah_item']].copy()
display_df['itemsets'] = display_df['itemsets'].apply(lambda x: ', '.join(list(x)))
display_df.head(6)


## 🎯 5. Mengekstrak Aturan Asosiasi (*Association Rules*) & Visualisasi

Sekarang, mari kita ubah *Frequent Itemsets* di atas menjadi **Aturan Aksi Bisnis** ($A ightarrow B$).  
Kita akan menyaring aturan dengan kriteria:
* **Minimum Confidence $\ge 60\%$** (Tingkat kepastian minimal 60%).
* **Minimum Lift Ratio $> 1.0$** (Hanya aturan yang memiliki asosiasi positif nyata!).


In [ ]:
# Ekstraksi Aturan Asosiasi
rules = association_rules(itemset_apriori, metric="confidence", min_threshold=0.6)

# Filter hanya yang memiliki nilai Lift > 1.0 (Asosiasi Positif Nyata)
rules_valid = rules[rules['lift'] > 1.0].sort_values(by='lift', ascending=False).reset_index(drop=True)

# Format kolom agar rapi dibaca manusia
rules_tampil = rules_valid[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rules_tampil['antecedents'] = rules_tampil['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_tampil['consequents'] = rules_tampil['consequents'].apply(lambda x: ', '.join(list(x)))

print("=== DAFTAR ATURAN ASOSIASI EMAS (SORTED BY LIFT) ===")
print(rules_tampil.round(3).to_string(index=False))


In [ ]:
# Visualisasi Scatter Plot: Support vs Confidence (Warna mewakili Lift Ratio)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    rules_valid['support'],
    rules_valid['confidence'],
    c=rules_valid['lift'],
    cmap='viridis',
    s=120,
    edgecolor='black',
    alpha=0.85
)

cbar = plt.colorbar(scatter)
cbar.set_label("Lift Ratio (Kekuatan Asosiasi)", fontweight='bold')

plt.title("Peta Aturan Asosiasi: Support vs Confidence", fontsize=12, fontweight='bold')
plt.xlabel("Support (Seberapa Sering Muncul)")
plt.ylabel("Confidence (Seberapa Yakin Kausalitasnya)")
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

print("💡 Cara Membaca Grafik:")
print("1. Titik di pojok kanan atas dengan warna kuning terang adalah ATURAN EMAS: Sering muncul (High Support), sangat pasti (High Confidence), dan ikatannya sangat kuat (High Lift)!")


## 💼 6. Rekomendasi Aksi Bisnis Nyata bagi Manajemen Ritel

Berdasarkan hasil aturan asosiasi di atas, seorang Data Miner dapat memberikan 3 rekomendasi taktis bernilai puluhan juta rupiah bagi pemilik supermarket:

---

### 1. ☕ Paket Bundling "Sarapan Kopi & Manis"
* **Aturan Terdeteksi**: `Gula Pasir ➔ Kopi Bubuk` (Confidence = 100%, Lift = 3.0!)
* **Aksi Bisnis**: Buat paket promo bundling: *"Beli 1 Kopi Bubuk + 1 Gula Pasir Hemat 15%"*. Pelanggan merasa hemat, sementara omzet kasir per struk (*basket size*) naik 2x lipat!

---

### 2. 🍞 Tata Letak Rak: Strategi "Lorong Pemisah"
* **Aturan Terdeteksi**: `Mentega ➔ Roti Tawar` (Confidence = 100%, Lift = 1.71)
* **Aksi Bisnis**: **JANGAN menaruh Mentega dan Roti tepat di rak yang sama!**  
  Taruh Roti Tawar di rak paling depan lorong, dan taruh Mentega di ujung lorong paling belakang. Karena pembeli Roti *hampir pasti* mencari Mentega, mereka terpaksa berjalan menyusuri lorong dan berpeluang melihat produk lain (*Impulse Buying*)!

---

### 3. 🧀 Upselling Kasir Otomatis
* **Aturan Terdeteksi**: `[Keju Cheddar, Susu Sapi] ➔ Roti Tawar` (Confidence = 100%)
* **Aksi Bisnis**: Pasang sistem notifikasi cerdas di layar kasir. Saat kasir memindai Keju dan Susu, layar kasir otomatis menampilkan: *"Tawarkan Roti Tawar diskon Rp 2.000 kepada pelanggan!"*


## 🧠 7. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep aturan asosiasi, lalu jalankan sel kode di bawah untuk mencocokkan logika Anda:

---
* **Soal 1**: Mengapa aturan asosiasi dengan nilai Confidence 95% belum tentu merupakan aturan yang bagus jika nilai Lift-nya hanya 0.98?
* **Soal 2**: Apa prinsip utama pemangkasan (*Pruning*) pada algoritma Apriori?
* **Soal 3**: Mengapa algoritma FP-Growth jauh lebih efisien dibandingkan Apriori pada kumpulan data transaksi yang sangat besar?
* **Soal 4**: Apa arti nyata dari metrik Support 0.05 pada data 10.000 transaksi kasir?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 7
kunci_jawaban_modul_7 = {
    "Soal 1": "Karena jika Lift < 1, artinya kedua barang tersebut tidak memiliki asosiasi positif (bahkan berkorelasi negatif/kanibal). Confidence tinggi hanya karena barang Consequent (B) memang sangat populer dibeli semua orang secara alami, bukan karena dipicu oleh barang A.",
    "Soal 2": "Prinsip Apriori: Jika suatu himpunan barang tidak memenuhi minimum support, maka seluruh kombinasi turunan/superset-nya pasti tidak memenuhi minimum support sehingga langsung dibuang tanpa perlu dihitung.",
    "Soal 3": "FP-Growth mengompresi seluruh transaksi ke dalam struktur pohon (FP-Tree) di memori sehingga hanya memindai data 2 kali, berbeda dengan Apriori yang memindai basis data berulang kali untuk setiap kandidat kombinasi.",
    "Soal 4": "Artinya kombinasi barang tersebut dibeli bersamaan pada 5% dari total transaksi, atau sebanyak 500 struk kasir."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 07 ===")
for soal, pembahasan in kunci_jawaban_modul_7.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 8. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Analisis Keranjang Apotek Farmasi!
Berikut adalah 6 transaksi kasir sebuah apotek:
* Transaksi 1: `['Paracetamol', 'Vitamin C', 'Masker Medis']`
* Transaksi 2: `['Paracetamol', 'Termometer']`
* Transaksi 3: `['Vitamin C', 'Madu', 'Masker Medis']`
* Transaksi 4: `['Paracetamol', 'Vitamin C', 'Madu']`
* Transaksi 5: `['Vitamin C', 'Masker Medis']`
* Transaksi 6: `['Paracetamol', 'Vitamin C', 'Madu', 'Masker Medis']`

Tugas Anda:
1. Ubah transaksi apotek di atas menjadi matriks biner One-Hot.
2. Cari aturan asosiasi dengan `min_support=0.3` dan temukan aturan dengan **Lift tertinggi**!


In [ ]:
# Solusi Tantangan Mandiri Apotek
transaksi_apotek = [
    ['Paracetamol', 'Vitamin C', 'Masker Medis'],
    ['Paracetamol', 'Termometer'],
    ['Vitamin C', 'Madu', 'Masker Medis'],
    ['Paracetamol', 'Vitamin C', 'Madu'],
    ['Vitamin C', 'Masker Medis'],
    ['Paracetamol', 'Vitamin C', 'Madu', 'Masker Medis']
]

te_apotek = TransactionEncoder()
df_apotek = pd.DataFrame(te_apotek.fit(transaksi_apotek).transform(transaksi_apotek), columns=te_apotek.columns_)

fi_apotek = apriori(df_apotek, min_support=0.3, use_colnames=True)
rules_apotek = association_rules(fi_apotek, metric="lift", min_threshold=1.0).sort_values(by='lift', ascending=False)

print("=== HASIL ATURAN ASOSIASI APOTEK (TOP 3 BY LIFT) ===")
hasil_ringkas = rules_apotek[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3).copy()
hasil_ringkas['antecedents'] = hasil_ringkas['antecedents'].apply(lambda x: ', '.join(list(x)))
hasil_ringkas['consequents'] = hasil_ringkas['consequents'].apply(lambda x: ', '.join(list(x)))
print(hasil_ringkas.round(2).to_string(index=False))

top_ant = hasil_ringkas.iloc[0]['antecedents']
top_con = hasil_ringkas.iloc[0]['consequents']
top_lift = hasil_ringkas.iloc[0]['lift']
print(f"\n💊 Aturan Terkuat: Pasien yang membeli [{top_ant}] memiliki kecenderungan {top_lift:.2f}x lipat lebih tinggi untuk membeli [{top_con}]!")


## 🎯 9. Checklist Kelulusan Modul 07 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami kisah klasik dan manfaat nyata *Market Basket Analysis* di dunia industri.
- [ ] Saya dapat membedakan dengan gamblang makna matematis dan bisnis dari **Support**, **Confidence**, dan **Lift Ratio**.
- [ ] Saya memahami mengapa nilai $\text{Lift} > 1$ adalah syarat mutlak aturan asosiasi yang positif dan bermanfaat.
- [ ] Saya bisa mengubah log transaksi belanja mentah menjadi matriks biner One-Hot menggunakan `TransactionEncoder`.
- [ ] Saya memahami filosofi pemangkasan (*Pruning*) pada algoritma **Apriori** dan keunggulan kecepatan **FP-Growth**.
- [ ] Saya mampu merancang strategi penataan rak dan paket diskon bundling berdasarkan aturan yang diekstrak.

---
🚀 **Selamat! Anda telah menaklukkan Modul 07 dengan luar biasa!**  
Di modul berikutnya (**Modul 08**), kita akan menjelajahi **Deteksi Anomali (Isolation Forest) & Dasar Text Mining (TF-IDF & Analisis Sentimen)** untuk mengolah data teks tak terstruktur dan menangkal kecurangan sistem!
